In [ ]:
# World University Rankings 2026

## Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

## Loading Dataset in Pandas DataFrame 

df = pd.read_csv(r"C:\Users\chara\Downloads\world_university_rankings_2026.csv")

# Shape of the dataset
df.shape

# Printing first 5 rows 
df.head()

# Printing last 5 rows
df.tail()

print(f"Universities  : {df['university'].nunique()}")

print(f"Countries     : {df['country'].nunique()}")

print(f"In all 3      : {df['ranked_in_all_3'].sum()}")

print(f"\nTop 5 by QS:")
print(df.nsmallest(5,'qs_rank_2026')[['university','country','qs_rank_2026','the_rank_2026','arwu_rank_2025']].to_string(index=False))


## Exploratory Data Analysis(EDA)

# ── Color Palette ──
BG    = "#f8f9ff"
BG2   = "#ffffff"
NAVY  = "#002147"
BLUE  = "#1a3a6b"
GOLD  = "#c8971a"
GREEN = "#15803d"
RED   = "#dc2626"
PURP  = "#7c3aed"
DARK  = "#0a0f2e"
MGRAY = "#475569"
LGRAY = "#94a3b8"
DGRAY = "#dbeafe"
TEAL  = "#0891b2"

REGION_COLORS = {
    "North America": NAVY,
    "Europe":        BLUE,
    "Asia":          TEAL,
    "Oceania":       GREEN,
    "Latin America": GOLD,
    "Africa":        RED,
    "Middle East":   PURP,
}


## Top 15 Universities 2026


# ====== COLORS (define if not already) ======
BG = "#0E1117"
BG2 = "#1A1F2B"
DARK = "white"
LGRAY = "#AAAAAA"
DGRAY = "#333333"

# Example REGION COLORS (modify if needed)
REGION_COLORS = {
    "North America": "#4E79A7",
    "Europe": "#F28E2B",
    "Asia": "#59A14F",
    "Oceania": "#E15759"
}

# ====== DATA ======
top15 = df.nsmallest(15, 'qs_rank_2026').copy()
top15 = top15.sort_values('qs_rank_2026', ascending=False)

# Short labels
top15['label'] = top15['university'].apply(
    lambda x: x[:25] + "..." if len(x) > 25 else x
)

# ====== FIGURE ======
fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Positions
y_pos = np.arange(len(top15))

# Colors
colors = top15['region'].map(REGION_COLORS).fillna(LGRAY)

# Highlight MIT (rank 1)
bar_colors = [
    "#FFD700" if rank == 1 else col
    for rank, col in zip(top15['qs_rank_2026'], colors)
]

# ====== PLOT ======
bars = ax.barh(
    y_pos,
    top15['qs_score'],
    color=bar_colors,
    height=0.6,
    edgecolor=BG2
)

# ====== FIX LABEL ISSUE (CRITICAL) ======
ax.set_yticks(y_pos)
ax.set_yticklabels(top15['label'], fontsize=10, color="white")

# ====== TEXT ON BARS ======
for i, (bar, (_, row)) in enumerate(zip(bars, top15.iterrows())):
    y = bar.get_y() + bar.get_height()/2

    # Score
    ax.text(
        bar.get_width() + 0.5,
        y,
        f"{row['qs_score']:.1f}",
        va='center',
        fontsize=9,
        color="white",
        fontweight='bold'
    )

    # Rank
    ax.text(
        1,
        y,
        f"#{int(row['qs_rank_2026'])}",
        va='center',
        fontsize=8,
        color="#BBBBBB",
        fontweight='bold'
    )

# ====== TITLE ======
ax.set_title(
    "🌍 Top 15 Universities — QS Rankings 2026\n"
    "🏆 MIT leads for the 14th consecutive year",
    fontsize=14,
    color="white",
    fontweight='bold',
    pad=20
)

# ====== STYLE ======
ax.set_xlabel("QS Score", color="white", fontsize=10)
ax.tick_params(axis='x', colors="white")

ax.grid(axis='x', linestyle='--', alpha=0.15)

# Remove spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Limits
ax.set_xlim(0, 107)

# ====== LEGEND ======
legend_items = [
    mpatches.Patch(color=color, label=region)
    for region, color in REGION_COLORS.items()
    if region in top15['region'].values
]

ax.legend(
    handles=legend_items,
    loc='lower right',
    frameon=False,
    fontsize=9,
    labelcolor='white'
)

# ====== IMPORTANT: SPACE FIX ======
plt.subplots_adjust(left=0.4)

plt.tight_layout()
plt.show()

# Region-wise Performance (Boxplot)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x='region',
    y='qs_score',
    palette='Set2'
)

plt.title("🌍 QS Score Distribution by Region", fontsize=14, fontweight='bold')
plt.xlabel("Region")
plt.ylabel("QS Score")

plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Top Countries Dominance (Bar Chart)

top100 = df.nsmallest(100, 'qs_rank_2026')

country_counts = top100['country'].value_counts().head(10)

plt.figure(figsize=(10,6))

bars = plt.bar(
    country_counts.index,
    country_counts.values
)

plt.title("🏆 Top Countries by Number of Top 100 Universities", fontsize=14, fontweight='bold')
plt.xlabel("Country")
plt.ylabel("Count")

plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Add values
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        int(bar.get_height()),
        ha='center',
        fontsize=10
    )

plt.tight_layout()
plt.show()

# Correlation (Heatmap)
import seaborn as sns

plt.figure(figsize=(8,6))

corr = df.select_dtypes(include='number').corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title("📊 Correlation Between QS Metrics", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

corr_target = df.select_dtypes(include='number').corr()['qs_score']

corr_target = corr_target.sort_values(ascending=False)

print(corr_target.round(2))


importance = pd.Series(model.feature_importances_, index=X.columns)
important_features = importance[importance > 0.01].index

X = X[important_features]


